# Introducción al Streaming y Flujos de Eventos

---

## ¿Qué es el Data Streaming?

El Data Streaming es el paradigma de procesar datos en tiempo real, gestionando flujos de datos no acotados (unbounded data).  
A diferencia de los conjuntos de datos tradicionales, un flujo es una secuencia infinita de mensajes generados gradualmente a lo largo del tiempo.

### ¿Qué es un Evento?

Un evento es la unidad básica de un flujo: un objeto pequeño, inmutable y autodescriptivo que registra que algo sucedió en un momento específico.  
Ejemplos:
- Acciones de usuario: clics, compras, vistas de página.
- Lecturas de sensores: temperatura, utilización de CPU.

---

## Batch vs. Streaming: El Cambio de Paradigma

| Batch | Streaming |
|-------|-----------|
| Procesa grandes volúmenes de datos acumulados en intervalos fijos. | Procesa datos en tiempo real, con baja latencia. |
| Latencia alta, permite análisis complejos. | Ideal para responder a eventos inmediatos. |
| Datos estáticos y almacenados (data-at-rest). | Datos que fluyen continuamente (data-in-motion). |
| Entrada finita y conocida. | Entrada infinita, el trabajo nunca termina. |
| Throughput como métrica principal. | Latencia como métrica principal. |

---

## Ciclo de Vida del Dato en Streaming

1. **Generación (Upstream):** Aplicaciones productoras generan el flujo de datos original (sensores, logs).
2. **Distribución (Message Processor):** Sistemas como Kafka organizan y distribuyen los mensajes.
3. **Procesamiento (Stream Processor):** Se aplica lógica de negocio: filtrado, transformación, joins, enriquecimiento.
4. **Consumo/Salida:** Resultados enviados a sistemas externos, alertas o tableros en tiempo real.

---

## ¿Qué es Kafka?

Kafka es una plataforma de mensajería distribuida, diseñada para el procesamiento de flujos de datos en tiempo real.  
Actúa como un broker, recibiendo, almacenando y distribuyendo eventos entre productores y consumidores.  
Ideal para sistemas de alta escala y baja latencia.

---

## Conceptos Clave del Streaming

- **Ventanas temporales (windows):** Agrupan datos en intervalos de tiempo para agregaciones.
- **Watermark:** Maneja datos tardíos, asegurando cierre correcto de ventanas.
- **Triggers:** Definen cuándo se ejecuta la agregación.
- **Estado (state):** Spark puede recordar resultados previos para cálculos acumulados.

---

## Structured Streaming en Spark

Spark Structured Streaming integra DataFrames con procesamiento continuo.  
Permite aplicar transformaciones similares a las de batch, pero en tiempo real.  
Usa ventanas, watermarks y checkpoints para asegurar consistencia.

---

## Ejemplo Práctico: Simulación de Kafka

Simulamos un flujo de eventos usando el source `rate`.  
Transformamos los datos simulados (usuario, producto, monto) y agregamos ventas por minuto.  
Estas métricas alimentan un modelo de Random Forest.  
En producción, solo cambiaríamos la fuente de datos (de rate a Kafka).

---

## Los Tres Pilares de las Aplicaciones de Datos

- **Confiabilidad:** El sistema debe funcionar correctamente ante fallas, garantizando procesamiento "exactamente una vez".
- **Escalabilidad:** Capacidad de manejar crecimiento en volumen o velocidad de datos sin degradar rendimiento.
- **Mantenibilidad:** Fácil de operar y evolucionar para nuevos requisitos de negocio.

---

## Conclusión

El streaming en Spark permite procesar datos en tiempo real, generando, transformando, agregando y modelando datos de manera continua.  
Aunque no tengamos Kafka en la versión free, la lógica es la misma y sienta las bases para un pipeline productivo y escalable.

# Structured Streaming en Spark: Detalle y Ejemplo Ilustrativo

---

## ¿Qué es Structured Streaming?

Structured Streaming es el motor de procesamiento de datos en tiempo real de Apache Spark, que permite trabajar con flujos de datos usando DataFrames y SQL, igual que en procesamiento batch, pero de manera continua.  
Permite aplicar transformaciones, agregaciones y modelos de machine learning sobre datos que llegan en tiempo real.

# Idea clave

Trata el streaming como una tabla infinita que se actualiza continuamente.

# ¿Cómo funciona?
Los datos llegan en flujo continuo
Spark los divide en micro-batches
Aplica transformaciones
Actualiza resultados en tiempo real

**Fuente de datos:** Puede ser archivos, Kafka, sockets, etc.
**Procesamiento:** Se aplican transformaciones como filtros, agregaciones, joins.
**Salida:** Los resultados pueden escribirse en consola, archivos, bases de datos o tablas Delta.


# Qué permite
Agregaciones en vivo 📊
Ventanas de tiempo ⏱️
Joins entre streams 🔗
Integración con Kafka, Delta Lake, etc.

---

## Ejemplo Ilustrativo: Streaming de Archivos JSON

Supongamos que tenemos un directorio donde se generan archivos JSON con datos de ventas. Queremos procesarlos en tiempo real y mostrar el resultado en consola.

### 1. Definir el esquema de los datos

python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("value", IntegerType(), True)
])


### 2. Leer los archivos en streaming

python
df = spark.readStream \
    .schema(schema) \
    .format("json") \
    .load("/Volumes/forecasting/default/data/")


### 3. Escribir el stream en consola

python
query = df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/forecasting/default/data/checkpoint_simple") \
    .trigger(availableNow=True) \
    .start()

query.awaitTermination()


---

## ¿Qué ocurre en este ejemplo?

- Spark detecta nuevos archivos JSON en el directorio y los procesa en tiempo real.
- Los datos se muestran en consola a medida que llegan.
- El checkpoint asegura tolerancia a fallos y seguimiento del progreso.

---

## Ventajas de Structured Streaming

- **Facilidad:** Usa la misma API que batch.
- **Escalabilidad:** Procesa grandes volúmenes de datos en tiempo real.
- **Confiabilidad:** Garantiza procesamiento exactamente una vez.
- **Integración:** Compatible con Delta Lake, Kafka, y otros sistemas.

---

## Resumen

Structured Streaming en Spark permite construir pipelines de datos en tiempo real de forma sencilla, robusta y escalable.  
El ejemplo muestra cómo leer archivos JSON en streaming, procesarlos y visualizar los resultados, sentando la base para aplicaciones más complejas como agregaciones, alertas o modelos de machine learning.

In [0]:
#Simulo los datos de entrada real time
import time
import json
import random
import os

path = "/Volumes/forecasting/default/data/"

for i in range(20):
    data = [{
        "user_id": random.randint(1, 5),
        "category": random.choice(["A", "B", "C"]),
        "value": random.randint(1, 100)
    }]
    
    file_path = path + f"data_{i}.json"
    
    with open(file_path, "w") as f:
        json.dump(data, f)
    
    print(f"Archivo generado: {file_path}")
    time.sleep(2)

In [0]:
#Define el esquema y lee los json como si fuera real time
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("value", IntegerType(), True)
])

df = spark.readStream \
    .schema(schema) \
    .format("json") \
    .load("/Volumes/forecasting/default/data/")


In [0]:
# Ejemplo simple de Structured Streaming

# Leer archivos JSON en streaming desde el directorio
df = spark.readStream \
    .schema(schema) \
    .format("json") \
    .load("/Volumes/forecasting/default/data/")

# Mostrar el stream en consola
query = df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/forecasting/default/data/checkpoint_simple") \
    .trigger(availableNow=True) \
    .start()

query.awaitTermination()

In [0]:
# Escribe el stream en consola, usando modo append y trigger availableNow
query = df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/forecasting/default/data") \
    .trigger(availableNow=True) \
    .start()

In [0]:
query


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lag, when, sum, window
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# Iniciamos la sesión de Spark
spark = SparkSession.builder.getOrCreate()

# 1. Generar stream artificial con rate (produce filas con timestamp y valor incremental)
df_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)  # Genera 5 filas por segundo
    .load()
)

# 2. Simular columnas de evento a partir del stream artificial
df_events = (
    df_stream
    .withColumn("user_id", (col("value") % 10).cast("int"))         # user_id simulado entre 0 y 9
    .withColumn("product_id", (col("value") % 5).cast("int"))       # product_id simulado entre 0 y 4
    .withColumn("amount", (col("value") % 100) + 1)                 # amount simulado entre 1 y 100
)

# 3. Agregación por minuto y producto: suma de ventas por ventana de 1 minuto
df_agg = (
    df_events
    .groupBy(
        window(col("timestamp"), "1 minute"),   # Ventana de 1 minuto basada en timestamp
        col("product_id")                      # Agrupa por product_id
    )
    .agg(
        sum("amount").alias("total_sales")     # Suma de amount por grupo
    )
)

# 4. Escribir los datos agregados a una tabla Delta usando streaming
query = (
    df_agg.writeStream
    .format("delta")                                              # Formato Delta Lake
    .outputMode("complete")                                       # Modo completo: sobrescribe toda la tabla
    .option("checkpointLocation", "/Volumes/forecasting/default/data/checkpoints/ml_training")  # Ubicación de checkpoint
    .trigger(availableNow=True)                                   # Procesa todos los datos disponibles y termina
    .toTable("forecasting.default.streaming_sales_agg")           # Escribe en la tabla Delta
)

# Esperar a que termine el procesamiento del stream
query.awaitTermination()

# 5. Leer los datos agregados como batch para transformaciones posteriores
df_batch = spark.read.table("forecasting.default.streaming_sales_agg")

# 6. Calcular ventas del minuto anterior usando función lag sobre ventana por producto
window_spec = Window.partitionBy("product_id").orderBy("window")
df_lag = (
    df_batch
    .withColumn("prev_total_sales", lag("total_sales").over(window_spec))  # Ventas del minuto anterior
)

# 7. Crear etiqueta binaria: 1 si suben las ventas respecto al minuto anterior, 0 si bajan o igual
df_lag = df_lag.withColumn(
    "label",
    when(col("prev_total_sales").isNull(), 0)                      # Si no hay dato previo, etiqueta 0
    .when(col("total_sales") > col("prev_total_sales"), 1)         # Si suben las ventas, etiqueta 1
    .otherwise(0)                                                  # Si bajan o igual, etiqueta 0
)

# 8. Ensamblar las características para ML: total_sales y prev_total_sales en un vector
assembler = VectorAssembler(inputCols=["total_sales", "prev_total_sales"], outputCol="features", handleInvalid="skip")

# 9. Dividir el dataset en train y test (70% train, 30% test)
train, test = df_lag.randomSplit([0.7, 0.3], seed=42)

# 10. Entrenar el modelo Random Forest usando las características ensambladas y la etiqueta
rf = RandomForestClassifier(featuresCol="features", labelCol="label")
pipeline = Pipeline(stages=[assembler, rf])
model = pipeline.fit(train)

In [0]:
# Iniciar el stream escribiendo a memoria para poder visualizarlo
query_stream = df_stream.writeStream \
    .format("memory") \
    .queryName("stream_data") \
    .outputMode("append")



In [0]:
query2 = (
    df_stream.writeStream
    .format("console")                          # Escribe el resultado en la consola
    .outputMode("append")                       # Solo agrega nuevas filas
    .option("truncate", "false")                # No corta las columnas largas
    .option("checkpointLocation", "/Volumes/forecasting/default/data/checkpoint_query2")
    .trigger(availableNow=True)                 # Procesa todo lo disponible y termina
    .start()                                    # Inicia el streaming query
)

query2.awaitTermination()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lag, when, sum, window
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# Iniciamos la sesión de Spark
spark = SparkSession.builder.getOrCreate()

# 1. Generar stream artificial con rate (produce filas con timestamp y valor incremental)
df_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)  # Genera 5 filas por segundo
    .load()
)

# 2. Simular columnas de evento a partir del stream artificial
df_events = (
    df_stream
    .withColumn("user_id", (col("value") % 10).cast("int"))         # user_id simulado entre 0 y 9
    .withColumn("product_id", (col("value") % 5).cast("int"))       # product_id simulado entre 0 y 4
    .withColumn("amount", (col("value") % 100) + 1)                 # amount simulado entre 1 y 100
)

# 3. Agregación por minuto y producto: suma de ventas por ventana de 1 minuto
df_agg = (
    df_events
    .groupBy(
        window(col("timestamp"), "1 minute"),   # Ventana de 1 minuto basada en timestamp
        col("product_id")                      # Agrupa por product_id
    )
    .agg(
        sum("amount").alias("total_sales")     # Suma de amount por grupo
    )
)

# 4. Escribir los datos agregados a una tabla Delta usando streaming
query = (
    df_agg.writeStream
    .format("delta")                                              # Formato Delta Lake
    .outputMode("complete")                                       # Modo completo: sobrescribe toda la tabla
    .option("checkpointLocation", "/Volumes/forecasting/default/data/checkpoints/ml_training")  # Ubicación de checkpoint
    .trigger(availableNow=True)                                   # Procesa todos los datos disponibles y termina
    .toTable("forecasting.default.streaming_sales_agg")           # Escribe en la tabla Delta
)

# Esperar a que termine el procesamiento del stream
query.awaitTermination()

# 5. Leer los datos agregados como batch para transformaciones posteriores
df_batch = spark.read.table("forecasting.default.streaming_sales_agg")

# 6. Calcular ventas del minuto anterior usando función lag sobre ventana por producto
window_spec = Window.partitionBy("product_id").orderBy("window")
df_lag = (
    df_batch
    .withColumn("prev_total_sales", lag("total_sales").over(window_spec))  # Ventas del minuto anterior
)

# 7. Crear etiqueta binaria: 1 si suben las ventas respecto al minuto anterior, 0 si bajan o igual
df_lag = df_lag.withColumn(
    "label",
    when(col("prev_total_sales").isNull(), 0)                      # Si no hay dato previo, etiqueta 0
    .when(col("total_sales") > col("prev_total_sales"), 1)         # Si suben las ventas, etiqueta 1
    .otherwise(0)                                                  # Si bajan o igual, etiqueta 0
)

# 8. Ensamblar las características para ML: total_sales y prev_total_sales en un vector
assembler = VectorAssembler(inputCols=["total_sales", "prev_total_sales"], outputCol="features", handleInvalid="skip")

# 9. Dividir el dataset en train y test (70% train, 30% test)
train, test = df_lag.randomSplit([0.7, 0.3], seed=42)

# 10. Entrenar el modelo Random Forest usando las características ensambladas y la etiqueta
rf = RandomForestClassifier(featuresCol="features", labelCol="label")
pipeline = Pipeline(stages=[assembler, rf])
model = pipeline.fit(train)

## EJEMPLOS NO SOPORTADOS EN SERVERLERSS!!!!!

In [0]:
# Ejemplo de Structured Streaming con outputMode "append"
df = spark.readStream \
    .schema(schema) \
    .format("json") \
    .load("/Volumes/forecasting/default/data/")

query_append = df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/forecasting/default/data/checkpoint_append") \
    .start()

# Ejemplo de uso de triggers: procesamiento cada 10 segundos
query_trigger = df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/forecasting/default/data/checkpoint_trigger") \
    .trigger(processingTime="10 seconds") \
    .start()

In [0]:
# Ejemplo de Structured Streaming usando trigger para controlar la frecuencia de procesamiento

df = spark.readStream \
    .schema(schema) \
    .format("json") \
    .load("/Volumes/forecasting/default/data/")

# Procesa el stream cada 10 segundos usando trigger
query = df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/forecasting/default/data/checkpoint_trigger_example") \
    .trigger(processingTime="10 seconds") \
    .start()

query.awaitTermination()